# DA6401 Report - Section 2.1: Data Exploration and Class Distribution

This notebook prepares the W&B artifacts needed for Assignment section 2.1:
- Class-count distribution table
- 5 sample images per class (10 classes)
- A gallery visualization for quick qualitative inspection


In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from keras.datasets import mnist
import wandb


In [ ]:
# Config
PROJECT = "da6401_assignment_1"
ENTITY = None  # Set to your W&B entity if needed
RUN_NAME = "report_2_1_data_exploration_mnist"
RUN_GROUP = "report_v1"
SEED = 42
SAMPLES_PER_CLASS = 5

RUN_EXPERIMENT = False  # Set True only when you intentionally want to log a new W&B run.


In [ ]:
# Load MNIST raw train split
(x_train, y_train), _ = mnist.load_data()

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)


In [ ]:
# Build deterministic per-class sample index list
rng = np.random.default_rng(SEED)
class_ids = list(range(10))
sample_indices = {}
for cls in class_ids:
    idx = np.where(y_train == cls)[0]
    sample_indices[cls] = rng.choice(idx, size=SAMPLES_PER_CLASS, replace=False)

class_counts = {int(cls): int(np.sum(y_train == cls)) for cls in class_ids}
class_counts


In [ ]:
# Create a 10x5 gallery figure (rows=classes, cols=samples)
fig, axes = plt.subplots(10, SAMPLES_PER_CLASS, figsize=(SAMPLES_PER_CLASS * 1.8, 14))
for row, cls in enumerate(class_ids):
    for col, idx in enumerate(sample_indices[cls]):
        axes[row, col].imshow(x_train[idx], cmap="gray")
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(f"Class {cls}", loc="left", fontsize=10)
fig.suptitle("MNIST: 5 samples per class", fontsize=14)
fig.tight_layout()
plt.show()


In [ ]:
if RUN_EXPERIMENT:
    # Log required section 2.1 artifacts to W&B
    run = wandb.init(
        project=PROJECT,
        entity=ENTITY,
        name=RUN_NAME,
        group=RUN_GROUP,
        tags=["report_v1", "report_section_2.1", "mnist", "data_exploration"],
        config={
            "dataset": "mnist",
            "report_section": "2.1",
            "samples_per_class": SAMPLES_PER_CLASS,
            "seed": SEED,
        },
    )
    # Table 1: class counts
    count_table = wandb.Table(columns=["class", "count"])
    for cls in class_ids:
        count_table.add_data(cls, class_counts[cls])
    # Table 2: 5 samples per class
    sample_table = wandb.Table(columns=["class", "sample_id", "image"])
    for cls in class_ids:
        for sid, idx in enumerate(sample_indices[cls]):
            sample_table.add_data(
                cls,
                sid,
                wandb.Image(x_train[idx], caption=f"class_{cls}_sample_{sid}"),
            )
    wandb.log({
        "analysis/class_distribution_table": count_table,
        "dataset/class_samples": sample_table,
        "analysis/class_sample_gallery": wandb.Image(fig),
    })
    run.summary["analysis/num_classes"] = 10
    run.summary["analysis/samples_per_class"] = SAMPLES_PER_CLASS
    run.finish()
    print("Logged section 2.1 artifacts to W&B.")
else:
    print('RUN_EXPERIMENT=False -> no W&B calls made.')


## Notes for report writing
- Import `dataset/class_samples` table directly into W&B Report section 2.1.
- Import `analysis/class_distribution_table` and `analysis/class_sample_gallery`.
- Add your qualitative note on visually similar classes (for MNIST, common pairs include 4/9, 3/5, and 5/8).
